###Import Libraries

In [1]:
import numpy as np
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

###Load Dataset

In [11]:
dataset = load_dataset("ag_news")

train_dataset = train_dataset.shuffle(seed=42).select(range(10000))
test_dataset = test_dataset.shuffle(seed=42).select(range(2000))

###Load Tokenizer And its function

In [12]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [13]:
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

In [14]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
test_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

###Load BERT Model

In [15]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


##Evaluation Metrics

In [16]:
def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

Training

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # reduce
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.403568,0.377056,0.910500,0.910109,0.911272,0.910500
2,0.233664,0.391455,0.919000,0.918945,0.919468,0.919000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5000, training_loss=0.345511083984375, metrics={'train_runtime': 2373.3857, 'train_samples_per_second': 8.427, 'train_steps_per_second': 2.107, 'total_flos': 5262315601920000.0, 'train_loss': 0.345511083984375, 'epoch': 2.0})

In [20]:
trainer.evaluate()

{'eval_loss': 0.39145514369010925,
 'eval_accuracy': 0.919,
 'eval_f1': 0.9189445880035987,
 'eval_precision': 0.9194675753929905,
 'eval_recall': 0.919,
 'eval_runtime': 66.9366,
 'eval_samples_per_second': 29.879,
 'eval_steps_per_second': 7.47,
 'epoch': 2.0}

In [21]:
trainer.save_model("news_classifier")
tokenizer.save_pretrained("news_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('news_classifier/tokenizer_config.json', 'news_classifier/tokenizer.json')